# Gradient Boosting Intuition

In this notebook, we'll demonstrate the core mechanism of Gradient Boosting Machines (GBM).
Instead of trying to predict the target $Y$ directly, each new tree in a GBM tries to predict the **errors** (residuals) of the combined previous trees.

We will use `sklearn`'s simple `DecisionTreeRegressor` as our "weak learner" to manually build a GBM step-by-step.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor

np.random.seed(42)

## 1. Create a Toy Dataset
We'll use a noisy sine wave. It's complex enough that a single weak tree won't fit it well, but an ensemble will.

In [ ]:
X = np.sort(5 * np.random.rand(80, 1), axis=0)
y = np.sin(X).ravel() + np.random.normal(0, 0.1, X.shape[0])

plt.scatter(X, y, color="darkorange", label="Data")
plt.title("Target Data")
plt.legend()
plt.show()

## 2. Step-by-Step Gradient Boosting

Let's define a very weak learner: A decision tree with a maximum depth of 1 (a "stump").

### Iteration 0: The Baseline
We start by predicting the mean of $y$ for all samples.

In [ ]:
F0 = np.mean(y)
predictions = np.full_like(y, F0)

plt.scatter(X, y, color="darkorange", label="Data")
plt.plot(X, predictions, color="navy", label="F0 (Mean)", linewidth=2)
plt.title("Iteration 0: Baseline Prediction")
plt.legend()
plt.show()

### Iteration 1
1. Calculate the residuals (errors) from our current predictions: `r1 = y - predictions`
2. Train a weak tree (`tree1`) to predict these residuals.
3. Update our predictions: `predictions = predictions + learning_rate * tree1.predict(X)`

In [ ]:
learning_rate = 0.5

# 1. Calculate residuals
r1 = y - predictions

# 2. Fit tree to residuals
tree1 = DecisionTreeRegressor(max_depth=1)
tree1.fit(X, r1)

# 3. Update predictions
predictions = predictions + learning_rate * tree1.predict(X)

# Plotting the new prediction
plt.scatter(X, y, color="darkorange", label="Data")
plt.plot(X, predictions, color="navy", label="F1 (F0 + tree1)", linewidth=2)
plt.title("Iteration 1")
plt.legend()
plt.show()

### Iteration 2
Repeat the process! Notice how the prediction gets slightly closer to the true data.

In [ ]:
# 1. Calculate residuals
r2 = y - predictions

# 2. Fit tree to residuals
tree2 = DecisionTreeRegressor(max_depth=1)
tree2.fit(X, r2)

# 3. Update predictions
predictions = predictions + learning_rate * tree2.predict(X)

plt.scatter(X, y, color="darkorange", label="Data")
plt.plot(X, predictions, color="navy", label="F2 (F1 + tree2)", linewidth=2)
plt.title("Iteration 2")
plt.legend()
plt.show()

## 3. Putting it in a Loop
Let's run this for 50 iterations and see the final result. We'll use a slightly deeper tree (`max_depth=2`) and a smaller learning rate (`0.1`) which is standard practice in GBMs to prevent overfitting.

In [ ]:
learning_rate = 0.1
n_estimators = 50

# Reset baseline
predictions = np.full_like(y, np.mean(y))
trees = []

for i in range(n_estimators):
    # Calculate residuals
    residuals = y - predictions
    
    # Fit weak learner to residuals
    tree = DecisionTreeRegressor(max_depth=2)
    tree.fit(X, residuals)
    trees.append(tree)
    
    # Update predictions
    predictions += learning_rate * tree.predict(X)

# Plot Final Ensemble
plt.figure(figsize=(10, 6))
plt.scatter(X, y, color="darkorange", label="Data", alpha=0.6)
plt.plot(X, predictions, color="firebrick", label=f"GBM ({n_estimators} trees)", linewidth=3)
plt.title("Final Gradient Boosting Machine Prediction")
plt.legend()
plt.show()

## Conclusion
You have just manually built a Gradient Boosting Machine!

**Why XGBoost?**
While the concept above is powerful, it is computationally slow on large datasets and prone to overfitting without careful tuning. XGBoost optimizes this exact process by:
1. **Second-Order Taylor Expansion**: Using both gradients (first derivative) and Hessians (second derivative) for faster and more accurate loss minimization.
2. **Regularization**: Adding penalties for tree complexity.
3. **Systems Optimization**: Cache-aware access, parallelized tree building, and handling out-of-core data.